# UT1.6 - Captura de datos desde JSON, Parquet y Excel

**Módulo:** Análisis de Datos con Python

**Resultado de Aprendizaje:** RA1 - Selecciona y utiliza las librerías principales de Python para la importación, manipulación, limpieza y exportación de datos.

**Nombre del Alumno/a:** ________________________________________

**Instrucciones:** Completa los bloques `# TU CÓDIGO AQUÍ` y ejecuta la celda de test correspondiente para comprobar tu solución.

In [ ]:
%reset -f
%run notebook_setup.py
%whos

## 1. Captura de datos desde JSON, Parquet y Excel (CE1.a, CE1.b, CE1.c)

El CSV no siempre es la mejor opción. Según el origen y el destino de los datos conviene elegir el formato:

| Formato | Naturaleza | Tipos | Anidamiento | Uso típico |
| :--- | :--- | :--- | :--- | :--- |
| **CSV** | Texto plano | No (todo texto) | No | Intercambio universal, hojas de cálculo |
| **JSON** | Texto jerárquico | No (números y texto) | **Sí** | APIs REST, documentos, configuración |
| **Parquet** | Binario columnar | **Sí** | Parcial | *Data lakes*, análisis masivo |
| **Excel** | Binario (`.xlsx`) | Parcial | No (varias hojas) | Informes para personas |

### 1.1. JSON: lectura, escritura y aplanado

`pd.read_json()` y `DataFrame.to_json()` trabajan con el parámetro `orient`, que indica cómo se estructura el texto:

| `orient` | Estructura |
| :--- | :--- |
| `"records"` | `[{"col": 1}, {"col": 2}]` (lista de objetos, lo más habitual en APIs) |
| `"columns"` | `{"col": {"0": 1, "1": 2}}` (columnas como claves) |
| `"index"` | `{"0": {"col": 1}, "1": {"col": 2}}` (índice como claves) |
| `"split"` | `{"index": [...], "columns": [...], "data": [...]}` |

Si el JSON tiene **claves anidadas**, `read_json()` las deja como diccionarios dentro de la celda.
Para convertirlas en columnas se usa `pd.json_normalize()`, que las aplana con notación de puntos (`cliente.nombre`).

In [ ]:
import json
import pathlib
import pandas as pd

pathlib.Path("data_sample").mkdir(exist_ok=True)

df_pedidos = pd.DataFrame({
    "pedido_id": [1, 2, 3],
    "cliente": ["Ana", "Luis", "Marta"],
    "importe": [120.0, 80.5, 45.0],
})

# DataFrame -> JSON (una lista de objetos)
df_pedidos.to_json("data_sample/pedidos.json", orient="records", force_ascii=False)
print(pathlib.Path("data_sample/pedidos.json").read_text(encoding="utf-8")[:200])

# JSON -> DataFrame
df_leido = pd.read_json("data_sample/pedidos.json", orient="records")
print("\n", df_leido)
print(df_leido.dtypes)

In [ ]:
# JSON anidado: el cliente lleva dentro su id, su nombre y su ciudad
pedidos_anidados = [
    {"pedido_id": 1, "importe": 120.0, "cliente": {"id": 10, "nombre": "Ana", "ciudad": "Madrid"}},
    {"pedido_id": 2, "importe": 80.0, "cliente": {"id": 11, "nombre": "Luis", "ciudad": "Sevilla"}},
]

# json_normalize aplana las claves anidadas con notación de puntos
df_plano = pd.json_normalize(pedidos_anidados)
print(df_plano)
print("\nColumnas:", df_plano.columns.tolist())

### 1.2. Parquet: binario columnar

Parquet guarda los datos **por columnas** y **con tipos**, comprimidos. Ventajas frente a CSV:

- Conserva los `dtypes` (fechas, enteros, categóricos) sin reparsear.
- Ocupa menos y se lee mucho más rápido.
- Permite leer **solo algunas columnas** (`columns=[...]`) sin cargar el resto.

Pandas lo soporta con `to_parquet()` / `read_parquet()`, pero necesita un motor: `pyarrow` (recomendado) o `fastparquet`.

In [ ]:
# DataFrame -> Parquet
df_pedidos.to_parquet("data_sample/pedidos.parquet", index=False)

# Parquet -> DataFrame leyendo solo dos columnas
df_parquet = pd.read_parquet("data_sample/pedidos.parquet", columns=["pedido_id", "importe"])
print(df_parquet)
print(df_parquet.dtypes)

print("\nTamaño CSV:", pathlib.Path("data_sample/pedidos.json").stat().st_size, "bytes")
print("Tamaño Parquet:", pathlib.Path("data_sample/pedidos.parquet").stat().st_size, "bytes")

### 1.3. Excel: varias hojas en un libro

`pd.read_excel()` lee una hoja (o todas con `sheet_name=None`).
Para escribir varias hojas en el mismo fichero se usa `pd.ExcelWriter` como gestor de contexto.

Requiere `openpyxl` instalado para `.xlsx`.

In [ ]:
with pd.ExcelWriter("data_sample/informe.xlsx") as writer:
    df_pedidos.to_excel(writer, sheet_name="pedidos", index=False)
    (
        df_pedidos.groupby("cliente", as_index=False)["importe"].sum()
        .to_excel(writer, sheet_name="resumen", index=False)
    )

# Hojas disponibles en el libro
print(pd.ExcelFile("data_sample/informe.xlsx").sheet_names)

# Lectura de una hoja concreta
print("\n", pd.read_excel("data_sample/informe.xlsx", sheet_name="resumen"))

### 1.4. Ejercicios

Los ejercicios escriben sus ficheros de prueba en `data_sample/`.
Necesitan `pyarrow` (Parquet) y `openpyxl` (Excel) instalados.

#### Ejercicio 1.1: Aplanado de JSON anidado (CE1.b)
Implementa `aplanar_pedidos(pedidos)` que reciba una lista de diccionarios con la clave anidada `cliente`
y devuelva un `DataFrame` plano con las columnas `pedido_id`, `importe`, `cliente.id` y `cliente.nombre`.

In [ ]:
def aplanar_pedidos(pedidos: list[dict]) -> pd.DataFrame:
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Implementa la función aplanar_pedidos")

In [ ]:
# TEST 1.1
pedidos = [
    {"pedido_id": 1, "importe": 120.0, "cliente": {"id": 10, "nombre": "Ana"}},
    {"pedido_id": 2, "importe": 80.0, "cliente": {"id": 11, "nombre": "Luis"}},
]

res_11 = aplanar_pedidos(pedidos)
assert list(res_11.columns) == ["pedido_id", "importe", "cliente.id", "cliente.nombre"]
assert res_11.loc[0, "cliente.nombre"] == "Ana"
assert res_11["importe"].sum() == 200.0
print("✅ Test Ejercicio 1.1 superado.")

#### Ejercicio 1.2: Conversión JSON → Parquet (CE1.b, CE1.c)
Implementa `json_a_parquet(ruta_json, ruta_parquet)` que:
1. Lea el JSON con `orient="records"`.
2. Lo exporte a Parquet sin índice.
3. Devuelva el número de filas exportadas.

In [ ]:
def json_a_parquet(ruta_json: str, ruta_parquet: str) -> int:
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Implementa la función json_a_parquet")

In [ ]:
# TEST 1.2
ruta_json = "data_sample/pedidos_test.json"
ruta_parquet = "data_sample/pedidos_test.parquet"

pd.DataFrame({
    "pedido_id": [1, 2, 3],
    "importe": [10.0, 25.5, 99.0],
}).to_json(ruta_json, orient="records")

filas = json_a_parquet(ruta_json, ruta_parquet)
df_res = pd.read_parquet(ruta_parquet)

assert filas == 3
assert list(df_res.columns) == ["pedido_id", "importe"]
assert df_res["importe"].sum() == 134.5
print("✅ Test Ejercicio 1.2 superado.")

#### Ejercicio 1.3: Exportación a Excel con varias hojas (CE1.c)
Implementa `exportar_resumen_excel(df, ruta_excel)` que escriba un libro con dos hojas:
- `datos`: el `DataFrame` completo, sin índice.
- `resumen`: el total de `importe` por `categoria`, sin índice y con `categoria` como columna.

In [ ]:
def exportar_resumen_excel(df: pd.DataFrame, ruta_excel: str) -> None:
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Implementa la función exportar_resumen_excel")

In [ ]:
# TEST 1.3
df_excel = pd.DataFrame({
    "categoria": ["Tech", "Hogar", "Tech"],
    "importe": [100.0, 50.0, 300.0],
})
ruta_excel = "data_sample/resumen_test.xlsx"

exportar_resumen_excel(df_excel, ruta_excel)
hojas = pd.ExcelFile(ruta_excel).sheet_names
resumen = pd.read_excel(ruta_excel, sheet_name="resumen")

assert hojas == ["datos", "resumen"]
assert resumen.loc[resumen["categoria"] == "Tech", "importe"].values[0] == 400.0
print("✅ Test Ejercicio 1.3 superado.")

#### Ejercicio 1.4: Lectura selectiva de Parquet (CE1.b)
Implementa `leer_columnas_parquet(ruta_parquet, columnas)` que devuelva únicamente las columnas indicadas,
aprovechando la lectura por columnas del formato.

In [ ]:
def leer_columnas_parquet(ruta_parquet: str, columnas: list[str]) -> pd.DataFrame:
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Implementa la función leer_columnas_parquet")

In [ ]:
# TEST 1.4
ruta_14 = "data_sample/columnas_test.parquet"
pd.DataFrame({
    "id": [1, 2],
    "categoria": ["A", "B"],
    "importe": [10.0, 20.0],
}).to_parquet(ruta_14, index=False)

res_14 = leer_columnas_parquet(ruta_14, ["id", "importe"])
assert list(res_14.columns) == ["id", "importe"]
assert len(res_14) == 2
print("✅ Test Ejercicio 1.4 superado.")